# A8 · DeepSeek-V4.1-Flash (CSA2 · Engram) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A8_deepseek_v41.ipynb)

원본: HF `deepseek-ai/DeepSeek-V4.1-Flash`의 `inference/model.py` + `engram.py` (커밋 `5254448` 고정).
transformers main에는 아직 `deepseek_v41`이 없어서 DeepSeek 공식 추론 코드를 받아 씁니다.
이 코드는 TileLang 커널을 쓰므로 **커널 함수만** 같은 수식의 fp32 PyTorch 구현으로 바꿔 끼웁니다
(fp8/fp4 양자화 흉내는 항등으로 둡니다. 자세한 목록은 `miniarch/verify/stages/a8_deepseek_v41.py` 맨 위 설명).

내 구현: `miniarch/models/deepseek_v41.py`. 가중치 이름은 실제 체크포인트 키와 같습니다.

V4에서 달라진 부품
- CSA2: sliding 층 + 압축 층. 압축 층은 모드가 셋입니다.
  Full(압축 KV와 indexer key를 직접 만듦) / Reindex(KV·key는 재사용, top-k만 다시 고름) / Reuse(선택까지 재사용)
- 2단계 indexer: ratio 1 묶음에서 candidate 출처 층이 block 단위로 먼저 추리고, 뒤 층은 그 안에서 top-k
- 압축: 토큰 r개를 softmax gate 가중합 (겹침·위치 bias 없음), r = 1이면 투영만. 압축 층 RoPE는 YaRN
- Engram: 2..n-gram 해시로 큰 표에서 벡터를 꺼내 줄기마다 gate를 걸어 residual에 더함
- mHC의 pre 계수가 한 sublayer 늦게 쓰임, MoE는 hash 라우팅 없이 sqrt-softplus + bias

축소 config (모든 경로가 한 번씩 쓰이도록 고름)
- 7층: [sliding, r2 Full, r2 Reindex, r2 Reuse, r1 Full(candidate 출처), r1 Reindex(candidate 사용), r1 Reuse]
- Engram은 1, 4번 층. 3-gram, head 2개. 토큰 정규화 표는 대소문자만 다른 128개 토큰짜리 작은 tokenizer로 만듦
- 시퀀스 41: window 8 밖, 홀수 길이라 ratio 2의 덜 찬 마지막 묶음, candidate block(4칸) 끝 padding까지 지나감
- index top-6 < 볼 수 있는 칸 수, candidate top-3 block < 전체 block 수 → 실제로 고르고 버리는 경로가 쓰임
- YaRN: original_seq_len 4096, factor 16 → rope 채널 일부가 보간 구간에 들어감
- 마지막 위치 logits와, 전 위치의 층별 출력(engram/attn/ffn/층)과 최종 norm 출력을 비교합니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 단계 전용 자체 점검 → 6. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip -q install -U -r requirements-colab.txt
# Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
!pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A8")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('deepseek-ai/DeepSeek-V4.1-Flash', '^layers\\.(1|2|24)\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 단계 전용 자체 점검

### 커널 대체 확인

원본 비교의 기준값이 커널 대체에 달려 있으므로, 대체한 `sparse_attn`이 정의(인덱스로 고른 칸 + sink에 대한 softmax)와 같은지,
`hc_split_sinkhorn`의 comb가 이중확률행렬인지 따로 확인합니다.

In [ ]:
K = S.torch_kernels()
V.set_deterministic(0)
b_, s_, h_, d_, n_, k_ = 2, 5, 3, 8, 12, 6
q_, kv_, sink_ = torch.randn(b_, s_, h_, d_), torch.randn(b_, n_, d_), torch.randn(h_)
idx_ = torch.stack([torch.randperm(n_)[:k_] for _ in range(b_ * s_)]).view(b_, s_, k_)
idx_[..., -2:] = -1
o1 = K.sparse_attn(q_, kv_, sink_, idx_.int(), 0.3)
mask_ = torch.zeros(b_, s_, n_ + 1, dtype=torch.bool).scatter_(-1, torch.where(idx_ >= 0, idx_, n_), True)[..., :n_]
lg_ = (torch.einsum("bshd,bnd->bshn", q_, kv_) * 0.3).masked_fill(~mask_[:, :, None], float("-inf"))
lg_ = torch.cat([lg_, sink_.view(1, 1, -1, 1).expand(b_, s_, h_, 1)], -1)
o2 = torch.einsum("bshn,bnd->bshd", lg_.softmax(-1)[..., :-1], kv_)
err = (o1 - o2).abs().max().item()
report("sparse_attn 대체 == 정의", err < 1e-5, err)

pre_, post_, comb_ = K.hc_split_sinkhorn(torch.randn(2, 3, 24), torch.rand(3) + 0.5, torch.randn(24) * 0.1, 4, 20, 1e-6)
err = max((comb_.sum(-1) - 1).abs().max().item(), (comb_.sum(-2) - 1).abs().max().item())
report("Sinkhorn comb 행/열 합 = 1", err < 1e-3, err)

### Engram n-gram 해시 id

해시는 학습 파라미터가 없는 전처리라 가중치 복사로는 검사되지 않습니다. 원본 `NgramHashState`와 내 `NgramHash`가
같은 정수 id를 내는지 직접 비교합니다 (소수 bucket 배치, 곱셈 계수, XOR 누적, pad 처리).

In [ ]:
from miniarch.models import deepseek_v41 as D41
ref, inp = S.ref()
m = D41.verify_pair(ref)["mine"]
ids = inp["input_ids"]
with torch.no_grad():
    a, b = ref.engram_hash(ids, 0), m.engram_hash(ids)
err = float((a - b).abs().max().item()) if a.shape == b.shape else float("nan")
print("hash ids", tuple(a.shape), "서로 다른 id 수:", a.unique().numel())
report("Engram 해시 id 일치", a.shape == b.shape and err == 0, err)

### 층 모드 · attention map · sink · Engram gate hook 확인

- 층마다 모드(sliding/full/reindex/reuse)와, 질의 하나가 평균 몇 개의 압축 칸을 보는지 출력합니다.
- `attn_probs`: [B, H, T, T + 압축 칸]. 각 행 합 + sink 확률 = 1이어야 합니다.
- `gate_probe`: Engram gate [B, T, 줄기].

In [ ]:
from miniarch.models import deepseek_v41 as D41
ref, inp = S.ref()
m = D41.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "sink_probe", "gate_probe"))]
x = inp["input_ids"].to(dev)
T_ = x.shape[1]
with V.capture(m, names) as st, torch.no_grad():
    m(x)
for i, layer in enumerate(m.layers):
    p, s = st[f"layers.{i}.attn.attn_probs"], st[f"layers.{i}.attn.sink_probe"]
    err = (p.sum(-1) + s - 1).abs().max().item()
    nc = (p[..., T_:] > 0).float().sum(-1).mean().item()
    print(f"layer {i} {layer.attn.mode:<8} ratio={layer.attn.ratio}  probs={tuple(p.shape)}  압축 칸 평균 {nc:.1f}개  평균 sink={s.mean():.3f}")
    report(f"layer {i} 확률 합", err < 1e-5, err)
for n in names:
    if n.endswith("gate_probe"):
        print(n, "평균 gate", round(st[n].mean().item(), 3))

### ablation 스위치

나중에 부품별 비교 실험에 쓸 스위치를 하나씩 끄고 forward가 정상(유한값)인지 확인합니다.

In [ ]:
import dataclasses
from miniarch.models import deepseek_v41 as D41
ref, inp = S.ref()
base = D41.config_from_hf(ref.args)
x = inp["input_ids"].to(dev)
for sw in ["use_engram", "use_mhc", "use_compressed_kv", "use_indexer", "use_candidates", "use_sink", "use_moe"]:
    m = D41.DeepseekV41(dataclasses.replace(base, **{sw: False})).to(dev).eval()
    with torch.no_grad():
        out = m(x).logits
    report(f"{sw}=False forward", bool(torch.isfinite(out).all()), 0.0)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A8_deepseek_v41.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A8_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")